# 🧰 1.2 Preprocesamiento de Entradas para el Pipeline

Este notebook prepara las entradas que consumirán los modelos de clasificación: organiza las secuencias, calcula/organiza los **embeddings ProtFlash**, construye las **máscaras de longitud**, genera el **embedding global/CLS** y serializa los conjuntos de datos (`.pt`) para entrenamiento, validación y test.

---

## 📋 Descripción general

A partir de las secuencias etiquetadas, el notebook produce las tensores que usan todas las arquitecturas del proyecto (MLP, CNN, BiLSTM, con/sin AAontology):

- `X`: embeddings por residuo de ProtFlash, con forma `(N, L, 768)`.
- `G`: embedding global/contextual (CLS), con forma `(N, 768)`.
- `M`: máscara binaria, con forma `(N, L)`, que marca residuos válidos vs. padding.
- `y`: etiquetas binarias.

Flujo principal:
1. Carga de secuencias y etiquetas.
2. Truncado/padding a una longitud máxima `L_MAX` y construcción de la máscara `M`.
3. Obtención de embeddings por residuo `X` y del embedding global `G`.
4. División en `train` / `val` / `test`.
5. Serialización de cada partición en archivos `.pt`.

---

## 🛠️ Funcionalidades principales

1. **Construcción de tensores**
   - Padding uniforme a `L_MAX` con máscara binaria asociada.
   - Extracción de embeddings por residuo y globales de ProtFlash.

2. **Particionado de datos**
   - División estratificada en entrenamiento, validación y test.
   - Verificación del balance de clases por partición.

3. **Serialización**
   - Exportación de `train_dataset.pt`, `val_dataset.pt` y `test_dataset.pt`.
   - Formato compatible con `torch.utils.data.Dataset`.

4. **Control de calidad**
   - Verificación de formas (shape checks).
   - Detección de secuencias anómalas o máscaras vacías.

## 📦 1. Instalación de dependencias

In [1]:
!apt-get update -qq
!apt-get install -qq cd-hit

# Verificar instalación
!cd-hit -h | head -n 3

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package cd-hit.
(Reading database ... 121026 files and directories currently installed.)
Preparing to unpack .../cd-hit_4.8.1-4_amd64.deb ...
Unpacking cd-hit (4.8.1-4) ...
Setting up cd-hit (4.8.1-4) ...
Processing triggers for man-db (2.10.2-1) ...
		====== CD-HIT version 4.8.1 (built on Aug 20 2021) ======

Usage: cd-hit [Options] 


## 📦 1. Importación de bibliotecas

In [2]:
import os
import glob
import re
import pandas as pd
import numpy as np
import subprocess
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')

## ⚙️ 2. Configuración de rutas y parámetros de validación

In [3]:
INPUT_CSV = "/kaggle/input/datasets/user/new-csv"
OUTPUT_CSV = "/kaggle/working/"
os.makedirs(OUTPUT_CSV, exist_ok=True)

VALID_AA_PATTERN = re.compile(r'^[ACDEFGHIKLMNPQRSTVWYX]+$')

CDHIT_THRESHOLD = 0.95

## 📂 3. Carga, etiquetado y validación por lotes

In [4]:
def load_and_clean_split(split_name: str, input_dir: str) -> pd.DataFrame:
    """
    Carga los CSV de un split (pos/neg), etiqueta, limpia y concatena.
    Usa operaciones vectorizadas para máxima velocidad.
    """
    frames = []
    
    for label, suffix in [(1, 'pos'), (0, 'neg')]:
        file_path = os.path.join(input_dir, f"{split_name}_{suffix}.csv")
        
        if not os.path.exists(file_path):
            print(f"⚠️ Archivo no encontrado: {file_path}")
            continue
            
        df = pd.read_csv(file_path)
        
        # 1. Etiquetado
        df['label'] = label
        
        # 2. Limpieza vectorizada (Mayúsculas y strip)
        df['sequence'] = df['sequence'].astype(str).str.upper().str.strip()
        
        # 3. Filtrado de aminoácidos no canónicos (Vectorizado con Regex)
        valid_mask = df['sequence'].str.match(VALID_AA_PATTERN)
        dropped = (~valid_mask).sum()
        df = df[valid_mask]
        
        # 4. Generación de ID único
        df['id'] = f"{split_name}_{suffix}_" + df.index.astype(str)
        
        frames.append(df[['id', 'sequence', 'label']])
        print(f"   📥 {split_name}_{suffix}: {len(df)} válidas | 🗑️ {dropped} descartadas (AA inválidos)")

    if not frames:
        return pd.DataFrame(columns=['id', 'sequence', 'label'])
        
    return pd.concat(frames, ignore_index=True)

## Filtrado de Redundancia con CD-HIT

In [5]:
def run_cdhit_intra_split(df: pd.DataFrame, split_name: str, threshold: float = 0.85) -> pd.DataFrame:
    """
    Ejecuta CD-HIT dentro de un solo split para eliminar redundancia interna.
    """
    if df.empty:
        return df
        
    in_f = f"temp_{split_name}.fasta"
    out_f = f"out_{split_name}.fasta"
    
    # 1. Exportar a FASTA (Vectorizado con NumPy)
    fasta_lines = (">" + df['id'] + "\n" + df['sequence']).values
    with open(in_f, 'w') as f:
        f.write("\n".join(fasta_lines) + "\n")
    
    # 2. Ejecutar CD-HIT
    cmd = f"cd-hit -i {in_f} -o {out_f} -c {threshold} -n 5 -d 0 -M 0 -T 1"
    try:
        subprocess.run(cmd, shell=True, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    except subprocess.CalledProcessError as e:
        print(f"❌ Error en CD-HIT para {split_name}: {e}")
        return df

    # 3. Leer IDs retenidos
    kept_ids = set()
    with open(out_f, 'r') as f:
        for line in f:
            if line.startswith('>'):
                kept_ids.add(line.strip()[1:])
                
    # 4. Filtrar DataFrame original
    original_len = len(df)
    df_filtered = df[df['id'].isin(kept_ids)].reset_index(drop=True)
    
    # 5. Limpieza de archivos temporales
    for ext in ['', '.clstr']:
        path = out_f + ext
        if os.path.exists(path): os.remove(path)
    if os.path.exists(in_f): os.remove(in_f)
    
    removed = original_len - len(df_filtered)
    print(f"   🧬 {split_name.upper()}: {original_len} → {len(df_filtered)} (🗑️ {removed} redundantes eliminadas)")
    
    return df_filtered

## 🚀 6. Ejecución del Pipeline y Exportación

In [6]:
splits_data = {}

print("📂 Fase 1: Carga, Etiquetado y Limpieza de AA")
print("=" * 50)
for split in ['train', 'val', 'test']:
    print(f"\n🔹 Procesando {split.upper()}...")
    df_split = load_and_clean_split(split, INPUT_CSV)
    splits_data[split] = df_split

print("\n\n🧬 Fase 2: Eliminación de Redundancia (CD-HIT Intra-Split)")
print("=" * 50)
for split in ['train', 'val']:
    splits_data[split] = run_cdhit_intra_split(splits_data[split], split, CDHIT_THRESHOLD)

print("\n\n💾 Fase 3: Aleatorización y Guardado")
print("=" * 50)
for split_name, df in splits_data.items():
    if df.empty:
        print(f"⚠️ {split_name.upper()} está vacío. Saltando...")
        continue
        
    # Shuffle reproducible
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)
    
    out_path = os.path.join(OUTPUT_CSV, f"{split_name}.csv")
    df.to_csv(out_path, index=False)
    
    n_total = len(df)
    n_pos = (df['label'] == 1).sum()
    n_neg = (df['label'] == 0).sum()
    ratio = n_pos / n_total if n_total > 0 else 0
    
    print(f"✅ {split_name.upper()}: {n_total} seqs | Pos: {n_pos} | Neg: {n_neg} | Ratio: {ratio:.3f}")
    print(f"   💾 Guardado en: {out_path}")

📂 Fase 1: Carga, Etiquetado y Limpieza de AA

🔹 Procesando TRAIN...
   📥 train_pos: 9781 válidas | 🗑️ 0 descartadas (AA inválidos)
   📥 train_neg: 9767 válidas | 🗑️ 0 descartadas (AA inválidos)

🔹 Procesando VAL...
   📥 val_pos: 2564 válidas | 🗑️ 0 descartadas (AA inválidos)
   📥 val_neg: 2561 válidas | 🗑️ 0 descartadas (AA inválidos)

🔹 Procesando TEST...
   📥 test_pos: 4914 válidas | 🗑️ 0 descartadas (AA inválidos)
   📥 test_neg: 10771 válidas | 🗑️ 0 descartadas (AA inválidos)


🧬 Fase 2: Eliminación de Redundancia (CD-HIT Intra-Split)
   🧬 TRAIN: 19548 → 13766 (🗑️ 5782 redundantes eliminadas)
   🧬 VAL: 5125 → 4128 (🗑️ 997 redundantes eliminadas)


💾 Fase 3: Aleatorización y Guardado
✅ TRAIN: 13766 seqs | Pos: 6114 | Neg: 7652 | Ratio: 0.444
   💾 Guardado en: /kaggle/working/train.csv
✅ VAL: 4128 seqs | Pos: 1939 | Neg: 2189 | Ratio: 0.470
   💾 Guardado en: /kaggle/working/val.csv
✅ TEST: 15685 seqs | Pos: 4914 | Neg: 10771 | Ratio: 0.313
   💾 Guardado en: /kaggle/working/test.csv


## 🏁 Resultado final: datasets serializados listos para entrenar

El notebook deja preparados los tres conjuntos de datos (entrenamiento, validación y test) en formato `.pt`, con embeddings ProtFlash, máscaras de longitud y etiquetas, listos para ser consumidos por todas las arquitecturas del proyecto.

### Entregables

| Entregable | Formato | Descripción |
|---|---|---|
| `train_dataset.pt` | PyTorch | Partición de entrenamiento |
| `val_dataset.pt` | PyTorch | Partición de validación |
| `test_dataset.pt` | PyTorch | Partición de test externo |

### Análisis

- **Consistencia:** las mismas tensores (`X`, `G`, `M`, `y`) alimentan de forma uniforme a los modelos Mean/Gated con y sin AAontology.
- **Reproducibilidad:** el particionado fijo (semilla estable) garantiza comparaciones justas entre arquitecturas.
- **Integridad:** los chequeos de forma y máscara evitan errores de padding aguas abajo.

**Estado final:** los datasets quedan serializados y disponibles para los notebooks de entrenamiento (MLP/CNN/BiLSTM) y para las validaciones externas.